In [2]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording


In [2]:
recording_list = ['20251212_128CHS_CA1_1_251212_164219', '20251212_128CHS_CA1_3_27_3g_251212_172228',
                  '20251212_128CHS_CA1_5_23_6g_251212_175849', '20251212_128CHS_V1_1_26g_251212_182717',
                  '20251212_128CHS_V1_2_22_4g_251212_184324', '20251212_128CHS_V1_3_27_1g_251212_185918']
channel_list = ['B-000', 'B-001', 'B-002',
       'B-003', 'B-004', 'B-005', 'B-006', 'B-007', 'B-008', 'B-009',
       'B-010', 'B-011', 'B-012', 'B-013', 'B-014', 'B-015', 'B-016',
       'B-017', 'B-018', 'B-019', 'B-020', 'B-021', 'B-022', 'B-023',
       'B-024', 'B-025', 'B-026', 'B-027', 'B-028', 'B-029', 'B-030',
       'B-031', 'B-032', 'B-033', 'B-034', 'B-035', 'B-036', 'B-037',
       'B-038', 'B-039', 'B-040', 'B-041', 'B-042', 'B-043', 'B-044',
       'B-045', 'B-046', 'B-047', 'B-048', 'B-049', 'B-050', 'B-051',
       'B-052', 'B-053', 'B-054', 'B-055', 'B-056', 'B-057', 'B-058',
       'B-059', 'B-060', 'B-061', 'B-062', 'B-063', 'B-064', 'B-065',
       'B-066', 'B-067', 'B-068', 'B-069', 'B-070', 'B-071', 'B-072',
       'B-073', 'B-074', 'B-075', 'B-076', 'B-077', 'B-078', 'B-079',
       'B-080', 'B-081', 'B-082', 'B-083', 'B-084', 'B-085', 'B-086',
       'B-087', 'B-088', 'B-089', 'B-090', 'B-091', 'B-092', 'B-093',
       'B-094', 'B-095', 'B-096', 'B-097', 'B-098', 'B-099', 'B-100',
       'B-101', 'B-102', 'B-103', 'B-104', 'B-105', 'B-106', 'B-107',
       'B-108', 'B-109', 'B-110', 'B-111', 'B-112', 'B-113', 'B-114',
       'B-115', 'B-116', 'B-117', 'B-118', 'B-119', 'B-120', 'B-121',
       'B-122', 'B-123', 'B-124', 'B-125', 'B-126', 'B-127']

In [5]:
for recording_name in recording_list:
    file_list = os.listdir(f"/home/ubuntu/Documents/jct/project/20251212-LN LYQ CA1 V1/{recording_name}")
    file_list.remove("settings.xml")

    file_list = sorted(file_list)
    recording_raw_list = []
    for file in file_list:
        recording_raw_list.append(se.read_intan(f"/home/ubuntu/Documents/jct/project/20251212-LN LYQ CA1 V1/{recording_name}/{file}", stream_id= '0'))
    recording_raw = concatenate_recordings(recording_list=recording_raw_list)
    recording_raw = recording_raw.select_channels(channel_list)
    
    recording_raw = spre.unsigned_to_signed(recording_raw)
    recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
    recording_recorded = spre.notch_filter(recording_recorded, freq=50)
    recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

    probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
    recording_f = recording_f.set_probegroup(probe)
    
    output_folder = f'/home/ubuntu/Documents/jct/project/sorted/20251212_LN_LYQ_CA1_V1/{recording_name}'
    recording_preprocessed = recording_f.save(format="binary", n_jobs = 5)

    sorting_kilosort4 = ss.run_sorter(
        sorter_name="kilosort4", 
        recording=recording_preprocessed, 
        folder=output_folder + "/kilosort4"
    )

    analyzer_kilosort4 = si.create_sorting_analyzer(
        sorting=sorting_kilosort4, 
        recording=recording_preprocessed, 
        format='binary_folder', 
        folder=output_folder + '/analyzer_kilosort4_binary'
    )

    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "noise_levels",
        "templates",
        "unit_locations",
        "spike_locations",
        "correlograms",
        "template_similarity"
    ]

    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "spike_locations": {"ms_before": 0.1},
        "correlograms": {"bin_ms": 0.1},
        "template_similarity": {"method": "cosine_similarity"}
    }

    analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params, n_jobs = 10)

    qm_params = sqm.get_default_qm_params()
    analyzer_kilosort4.compute("quality_metrics", qm_params, n_jobs = 10)

    sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)

Use cache_folder=/tmp/spikeinterface_cache/tmp_lm20z7a/IW7O8GYF
write_binary_recording 
engine=process - n_jobs=5 - samples_per_chunk=20,000 - chunk_memory=4.88 MiB - total_memory=24.41 MiB - chunk_duration=1.00s


write_binary_recording (workers: 5 processes):   0%|          | 0/679 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:51<00:00,  6.39s/it]


estimate_sparsity (no parallelization):   0%|          | 0/679 [00:00<?, ?it/s]

compute_waveforms (workers: 10 processes):   0%|          | 0/679 [00:00<?, ?it/s]

...

spike_amplitudes (no parallelization):   0%|          | 0/327 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/232 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/232 [00:00<?, ?it/s]

extract PCs (no parallelization):   0%|          | 0/327 [00:00<?, ?it/s]

Run:
phy template-gui  /home/ubuntu/Documents/jct/project/sorted/20251212_LN_LYQ_CA1_V1/20251212_128CHS_V1_3_27_1g_251212_185918/phy_folder_for_kilosort/params.py


In [4]:
file_list = os.listdir(f"/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1B_natima_251214_154409/")
file_list.remove("settings.xml")
file_list.remove("camera1_record_20251214_154447.mp4")
file_list.remove("camera2_record_20251214_154447.mp4")
file_list.remove("log_160509.csv")

file_list = sorted(file_list)
recording_raw_list = []
for file in file_list:
    recording_raw_list.append(se.read_intan(f"/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1B_natima_251214_154409/{file}", stream_id= '0'))
recording_raw = concatenate_recordings(recording_list=recording_raw_list)
recording_raw = recording_raw.select_channels(channel_list)

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
recording_f = recording_f.set_probegroup(probe)

output_folder = f'/media/ubuntu/sda/mouse_test/sorted/WLF_128ch2mouse&V1B_natima_251214_154409'
recording_preprocessed = recording_f.save(format="binary", n_jobs = 5)

sorting_kilosort4 = ss.run_sorter(
    sorter_name="kilosort4", 
    recording=recording_preprocessed, 
    folder=output_folder + "/kilosort4"
)

analyzer_kilosort4 = si.create_sorting_analyzer(
    sorting=sorting_kilosort4, 
    recording=recording_preprocessed, 
    format='binary_folder', 
    folder=output_folder + '/analyzer_kilosort4_binary'
)

extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "noise_levels",
    "templates",
    "unit_locations",
    "spike_locations",
    "correlograms",
    "template_similarity"
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "spike_locations": {"ms_before": 0.1},
    "correlograms": {"bin_ms": 0.1},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params, n_jobs = 10)

qm_params = sqm.get_default_qm_params()
analyzer_kilosort4.compute("quality_metrics", qm_params, n_jobs = 10)

sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)

Use cache_folder=/tmp/spikeinterface_cache/tmpb9so0wbk/O0STAMZE
write_binary_recording 
engine=process - n_jobs=5 - samples_per_chunk=20,000 - chunk_memory=4.88 MiB - total_memory=24.41 MiB - chunk_duration=1.00s


write_binary_recording (workers: 5 processes):   0%|          | 0/1588 [00:00<?, ?it/s]

100%|██████████| 8/8 [01:37<00:00, 12.23s/it]


estimate_sparsity (no parallelization):   0%|          | 0/1588 [00:00<?, ?it/s]

compute_waveforms (workers: 10 processes):   0%|          | 0/1588 [00:00<?, ?it/s]

...

spike_amplitudes (no parallelization):   0%|          | 0/1588 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/110 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/110 [00:00<?, ?it/s]

extract PCs (no parallelization):   0%|          | 0/1588 [00:00<?, ?it/s]

Run:
phy template-gui  /media/ubuntu/sda/mouse_test/sorted/WLF_128ch2mouse&V1B_natima_251214_154409/phy_folder_for_kilosort/params.py
